In [1]:
!pip install nltk scikit-learn

In [2]:
import nltk
import random
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB


In [4]:
!wget http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip

--2026-03-12 05:23:03--  http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
Resolving www.cs.cornell.edu (www.cs.cornell.edu)... 132.236.207.53
Connecting to www.cs.cornell.edu (www.cs.cornell.edu)|132.236.207.53|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip [following]
--2026-03-12 05:23:03--  https://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip
Connecting to www.cs.cornell.edu (www.cs.cornell.edu)|132.236.207.53|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9916637 (9.5M) [application/zip]
Saving to: ‘cornell_movie_dialogs_corpus.zip’

cornell_movie_dialo 100%[===================>]   9.46M  13.6MB/s    in 0.7s    

2026-03-12 05:23:04 (13.6 MB/s) - ‘cornell_movie_dialogs_corpus.zip’ saved [9916637/9916637]



In [5]:
!unzip cornell_movie_dialogs_corpus.zip

Archive:  cornell_movie_dialogs_corpus.zip
   creating: cornell movie-dialogs corpus/
  inflating: cornell movie-dialogs corpus/.DS_Store  
   creating: __MACOSX/
   creating: __MACOSX/cornell movie-dialogs corpus/
  inflating: __MACOSX/cornell movie-dialogs corpus/._.DS_Store  
  inflating: cornell movie-dialogs corpus/chameleons.pdf  
  inflating: __MACOSX/cornell movie-dialogs corpus/._chameleons.pdf  
  inflating: cornell movie-dialogs corpus/movie_characters_metadata.txt  
  inflating: cornell movie-dialogs corpus/movie_conversations.txt  
  inflating: cornell movie-dialogs corpus/movie_lines.txt  
  inflating: cornell movie-dialogs corpus/movie_titles_metadata.txt  
  inflating: cornell movie-dialogs corpus/raw_script_urls.txt  
  inflating: cornell movie-dialogs corpus/README.txt  
  inflating: __MACOSX/cornell movie-dialogs corpus/._README.txt  


In [1]:
!ls


'cornell movie-dialogs corpus'	    __MACOSX
 cornell_movie_dialogs_corpus.zip   sample_data


In [2]:
folder = "cornell movie-dialogs corpus"
!ls "{folder}"

chameleons.pdf		       movie_lines.txt		  README.txt
movie_characters_metadata.txt  movie_titles_metadata.txt
movie_conversations.txt        raw_script_urls.txt


In [3]:
lines_file = "cornell movie-dialogs corpus/movie_lines.txt"
conversations_file = "cornell movie-dialogs corpus/movie_conversations.txt"

# Check if they exist
!ls "cornell movie-dialogs corpus"

chameleons.pdf		       movie_lines.txt		  README.txt
movie_characters_metadata.txt  movie_titles_metadata.txt
movie_conversations.txt        raw_script_urls.txt


In [4]:
lines_file = "cornell movie-dialogs corpus/movie_lines.txt"

# Create a dictionary mapping line ID → text
lines = {}

with open(lines_file, encoding='latin-1') as f:
    for line in f:
        parts = line.split(" +++$+++ ")
        if len(parts) == 5:
            lines[parts[0]] = parts[4].strip()

print("Total lines loaded:", len(lines))
# Example
list(lines.items())[:5]

Total lines loaded: 304713


[('L1045', 'They do not!'),
 ('L1044', 'They do to!'),
 ('L985', 'I hope so.'),
 ('L984', 'She okay?'),
 ('L925', "Let's go.")]

In [5]:
conversations_file = "cornell movie-dialogs corpus/movie_conversations.txt"

conversations = []

with open(conversations_file, encoding='latin-1') as f:
    for line in f:
        parts = line.split(" +++$+++ ")
        # parts[3] is a list of line IDs like "['L194', 'L195']"
        line_ids = eval(parts[3])
        conversations.append(line_ids)

print("Total conversations loaded:", len(conversations))

Total conversations loaded: 83097


In [6]:
pairs = []

for conv in conversations:
    for i in range(len(conv) - 1):
        input_line = lines[conv[i]]
        target_line = lines[conv[i + 1]]
        pairs.append((input_line, target_line))

print("Total pairs:", len(pairs))
# Check a few examples
pairs[:5]

Total pairs: 221616


[('Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again.',
  "Well, I thought we'd start with pronunciation, if that's okay with you."),
 ("Well, I thought we'd start with pronunciation, if that's okay with you.",
  'Not the hacking and gagging and spitting part.  Please.'),
 ('Not the hacking and gagging and spitting part.  Please.',
  "Okay... then how 'bout we try out some French cuisine.  Saturday?  Night?"),
 ("You're asking me out.  That's so cute. What's your name again?",
  'Forget it.'),
 ("No, no, it's my fault -- we didn't have a proper introduction ---",
  'Cameron.')]

In [7]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9?.!,]+", " ", text)
    return text

pairs = [(clean_text(q), clean_text(a)) for q, a in pairs]
pairs[:5]

[('can we make this quick? roxanne korrine and andrew barrett are having an incredibly horrendous public break up on the quad. again.',
  'well, i thought we d start with pronunciation, if that s okay with you.'),
 ('well, i thought we d start with pronunciation, if that s okay with you.',
  'not the hacking and gagging and spitting part. please.'),
 ('not the hacking and gagging and spitting part. please.',
  'okay... then how bout we try out some french cuisine. saturday? night?'),
 ('you re asking me out. that s so cute. what s your name again?',
  'forget it.'),
 ('no, no, it s my fault we didn t have a proper introduction ', 'cameron.')]

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Separate questions and answers
questions = [p[0] for p in pairs]
answers = [p[1] for p in pairs]

# Convert text to vectors
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(questions)

# Train a simple NearestNeighbors model
model = NearestNeighbors(n_neighbors=1)
model.fit(X)

NearestNeighbors(n_neighbors=1)

In [9]:
def chatbot(message):
    msg_vec = vectorizer.transform([message])
    dist, idx = model.kneighbors(msg_vec)
    return answers[idx[0][0]]

In [12]:
chatbot("hello")


'finally! so you re our canadian pickpocket?'

In [ ]:
while True:
    msg = input("You: ")
    print("Bot:", chatbot(msg))

You: hii
Bot: hey.
You: how r u
Bot: could be raining!
You: did u eat?
Bot: nah. i m not hungry. i m sorry i didn t call. it was just, you know, hard to get away.
You: ok bye
Bot: maybe she has caller id.
You: bye
Bot: monica!
